In [ ]:
!nvidia-smi

# Multiple Optimizer Development

This notebook is for developing and testing using Muon along with AdamW for the linear and conv layers. Last updated 01-15-26.

In [ ]:
import json
import os
import pathlib
import socket
# from argparse import ArgumentParser

import torch
import yaml
from lightning import Trainer, seed_everything
from lightning.pytorch.callbacks import ModelCheckpoint

from src.models.lipsvision import get_module


# PyTorch 2.9+ matmul/cudnn precision: use ONLY the new API.
torch.set_float32_matmul_precision("medium")
# Do NOT set torch.backends.cuda/cudnn.allow_tf32 directly; 'set_float32_matmul_precision' handles it.

In [ ]:
hostname = socket.gethostname()

In [ ]:
seed = 42
config = "configs/lipsalexnet_mult_opt_v1/lipsalexnet_w_max_6.json"
num_workers = 2
gpus = 1
exp_dir = pathlib.Path("experiments")
resume_training = False
arg_ckpt_path = ""
num_nodes = 1

In [ ]:
seed_everything(seed)

In [ ]:
config_path = config
print(f"Loading config from {config_path}")

if config_path.endswith(".json"):
    with open(config_path, "r") as f:
        config = json.load(f)
else:
    with open(config_path, "r") as f:
        config = yaml.load(f, Loader=yaml.FullLoader)

In [ ]:
module = get_module(config)
print(f"Module: {module}")

In [ ]:
config["num_workers"] = num_workers
config["ngpus"] = gpus

In [ ]:
config_path = pathlib.Path(config_path)
checkpoint_dir = exp_dir / f"{config_path.stem}/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)

In [ ]:
ckpt_path = None
if resume_training and (len(ckpt_paths) > 0 or arg_ckpt_path != ""):
    if arg_ckpt_path != "":
        ckpt_path = arg_ckpt_path
        model = module.load_from_checkpoint(ckpt_path, config=config)
    else:
        ckpt_path = ckpt_paths[-1]
        model = module.load_from_checkpoint(ckpt_path, config=config)
else:
    model = module(config)

In [ ]:
trainer = Trainer(
    precision="32",
    default_root_dir=exp_dir / config_path.stem,
    max_epochs=config["hparams"]["max_epochs"],
    num_nodes=num_nodes,
    devices=gpus,
    accelerator="gpu",
    limit_val_batches=config["hparams"].get("limit_val_batches", 1.0),
    limit_train_batches=config["hparams"].get("limit_train_batches", 1.0),
    val_check_interval=config["hparams"].get("val_check_interval", 1.0),
    gradient_clip_val=config["hparams"].get("gradient_clip_val", None),
    gradient_clip_algorithm=config["hparams"].get("gradient_clip_algorithm", "value"),
    accumulate_grad_batches=config["hparams"].get("accumulate_grad_batches", 1),
    profiler=config["hparams"].get("profiler", None),
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
            save_last=True,
            save_weights_only=True,
        )
    ],
)

In [ ]:
trainer.fit(model)